In [0]:
print("Cart to Door Project")
print("Spark Version:", spark.version)

display(spark.sql("SHOW CATALOGS"))

Cart to Door Project
Spark Version: 4.2.0


catalog
samples
system
workspace


In [0]:
catalog = "workspace"
schema = "default"
volume = "olist_raw_data"

base_path = f"/Volumes/{catalog}/{schema}/{volume}"

print("Data location:", base_path)


Data location: /Volumes/workspace/default/olist_raw_data


In [0]:

from pyspark.sql import functions as F
import os

files = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

datasets = {}

for file in files:
    name = file.replace(".csv", "").replace("olist_", "")
    df = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/{file}")
    datasets[name] = df
    print(name, ":", df.count(), "rows")

print("All 9 CSV files loaded successfully!")


customers_dataset : 99441 rows
geolocation_dataset : 1000163 rows
order_items_dataset : 112650 rows
order_payments_dataset : 103886 rows
order_reviews_dataset : 104162 rows
orders_dataset : 99450 rows
products_dataset : 32951 rows
sellers_dataset : 3095 rows
product_category_name_translation : 71 rows
All 9 CSV files loaded successfully!


In [0]:

for name, df in datasets.items():
    table_name = f"bronze_{name}"
    
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"workspace.default.{table_name}")
    
    print(f"Created table: {table_name}")

print("BRONZE LAYER COMPLETED SUCCESSFULLY!")


Created table: bronze_customers_dataset
Created table: bronze_geolocation_dataset
Created table: bronze_order_items_dataset
Created table: bronze_order_payments_dataset
Created table: bronze_order_reviews_dataset
Created table: bronze_orders_dataset
Created table: bronze_products_dataset
Created table: bronze_sellers_dataset
Created table: bronze_product_category_name_translation
BRONZE LAYER COMPLETED SUCCESSFULLY!


In [0]:

from pyspark.sql import functions as F

silver_datasets = {}

for name, df in datasets.items():
    cleaned_df = df.dropDuplicates()
    
    # Remove completely empty rows
    cleaned_df = cleaned_df.na.drop(how="all")
    
    silver_datasets[name] = cleaned_df
    
    table_name = f"silver_{name}"
    
    cleaned_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"workspace.default.{table_name}")
    
    print(
        f"{table_name} | Original: {df.count()} | Cleaned: {cleaned_df.count()}"
    )

print("SILVER LAYER COMPLETED SUCCESSFULLY!")


silver_customers_dataset | Original: 99441 | Cleaned: 99441
silver_geolocation_dataset | Original: 1000163 | Cleaned: 738332
silver_order_items_dataset | Original: 112650 | Cleaned: 112650
silver_order_payments_dataset | Original: 103886 | Cleaned: 103886
silver_order_reviews_dataset | Original: 104162 | Cleaned: 104077
silver_orders_dataset | Original: 99450 | Cleaned: 99442
silver_products_dataset | Original: 32951 | Cleaned: 32951
silver_sellers_dataset | Original: 3095 | Cleaned: 3095
silver_product_category_name_translation | Original: 71 | Cleaned: 71
SILVER LAYER COMPLETED SUCCESSFULLY!


In [0]:

from pyspark.sql import functions as F

# 1. Monthly Orders Summary
orders = silver_datasets["orders_dataset"]

orders_with_month = orders.withColumn(
    "order_month",
    F.date_format(
        F.coalesce(
            F.to_timestamp("order_purchase_timestamp", "M/d/yyyy H:mm"),
            F.to_timestamp("order_purchase_timestamp", "MM/dd/yyyy HH:mm:ss"),
            F.to_timestamp("order_purchase_timestamp", "yyyy-MM-dd HH:mm:ss")
        ),
        "yyyy-MM"
    )
)

monthly_orders = (
    orders_with_month
    .filter(F.col("order_month").isNotNull())
    .groupBy("order_month")
    .agg(F.countDistinct("order_id").alias("total_orders"))
    .orderBy("order_month")
)

monthly_orders.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_monthly_orders"
)

print("1. Monthly Orders Summary Created")


# 2. Order Status Summary
order_status = (
    orders
    .groupBy("order_status")
    .agg(F.countDistinct("order_id").alias("total_orders"))
)

order_status.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_order_status"
)

print("2. Order Status Summary Created")


# 3. Payment Method Summary
payments = silver_datasets["order_payments_dataset"]

payment_summary = (
    payments
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("payment_records"),
        F.round(F.sum("payment_value"), 2).alias("total_payment_value")
    )
    .orderBy(F.desc("total_payment_value"))
)

payment_summary.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_payment_summary"
)

print("3. Payment Summary Created")


# 4. Product Category Summary
items = silver_datasets["order_items_dataset"]
products = silver_datasets["products_dataset"]
translation = silver_datasets["product_category_name_translation"]

category_summary = (
    items
    .join(products, "product_id", "left")
    .join(translation, "product_category_name", "left")
    .groupBy("product_category_name_english")
    .agg(
        F.count("*").alias("total_items_sold"),
        F.round(F.sum("price"), 2).alias("total_sales")
    )
    .orderBy(F.desc("total_sales"))
)

category_summary.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_category_summary"
)

print("4. Product Category Summary Created")

print("\nGOLD LAYER COMPLETED SUCCESSFULLY!")


1. Monthly Orders Summary Created
2. Order Status Summary Created
3. Payment Summary Created
4. Product Category Summary Created

GOLD LAYER COMPLETED SUCCESSFULLY!


In [0]:

from pyspark.sql import functions as F

# Helper function for safe date conversion
def parse_date(column_name):
    return F.coalesce(
        F.expr(f"try_to_timestamp(`{column_name}`, 'M/d/yyyy H:mm')"),
        F.expr(f"try_to_timestamp(`{column_name}`, 'MM/dd/yyyy HH:mm:ss')"),
        F.expr(f"try_to_timestamp(`{column_name}`, 'yyyy-MM-dd HH:mm:ss')"),
        F.expr(f"try_to_timestamp(`{column_name}`, 'yyyy-MM-dd HH:mm')")
    )

# 1. Delivery Performance Analysis
orders = silver_datasets["orders_dataset"]

delivery = (
    orders
    .withColumn("purchase_date", parse_date("order_purchase_timestamp"))
    .withColumn("delivered_date", parse_date("order_delivered_customer_date"))
    .withColumn("estimated_date", parse_date("order_estimated_delivery_date"))
    .filter(F.col("order_status") == "delivered")
    .withColumn(
        "delivery_days",
        F.datediff(F.to_date("delivered_date"), F.to_date("purchase_date"))
    )
    .withColumn(
        "delivery_performance",
        F.when(F.col("delivered_date").isNull(), "Unknown")
         .when(F.col("delivered_date") > F.col("estimated_date"), "Late")
         .otherwise("On Time or Early")
    )
)

delivery_summary = (
    delivery
    .groupBy("delivery_performance")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.avg("delivery_days"), 2).alias("average_delivery_days")
    )
)

delivery_summary.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_delivery_performance"
)

print("1. Delivery Performance Created")


# 2. Customer Review Summary
reviews = silver_datasets["order_reviews_dataset"]

reviews_clean = (
    reviews
    .withColumn(
        "review_score_numeric",
        F.expr("try_cast(trim(review_score) AS DOUBLE)")
    )
    .filter(F.col("review_score_numeric").between(1, 5))
)

review_summary = (
    reviews_clean
    .groupBy("review_score_numeric")
    .agg(F.count("*").alias("total_reviews"))
    .withColumnRenamed("review_score_numeric", "review_score")
    .orderBy("review_score")
)

review_summary.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_review_summary"
)

print("2. Customer Review Summary Created")


# 3. Delivery Delay vs Customer Ratings
delivery_reviews = (
    delivery
    .join(
        reviews_clean.select("order_id", "review_score_numeric"),
        on="order_id",
        how="inner"
    )
    .groupBy("delivery_performance")
    .agg(
        F.count("review_score_numeric").alias("total_reviews"),
        F.round(F.avg("review_score_numeric"), 2).alias("average_review_rating")
    )
)

delivery_reviews.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_delivery_review_summary"
)

print("3. Delivery vs Reviews Analysis Created")

print("\nDELIVERY AND REVIEW ANALYSIS COMPLETED!")


1. Delivery Performance Created
2. Customer Review Summary Created
3. Delivery vs Reviews Analysis Created

DELIVERY AND REVIEW ANALYSIS COMPLETED!


In [0]:

tables = spark.sql("SHOW TABLES IN workspace.default")

display(tables)

print("TOTAL TABLES:", tables.count())


database,tableName,isTemporary
default,bronze_customers_dataset,false
default,bronze_geolocation_dataset,false
default,bronze_order_items_dataset,false
default,bronze_order_payments_dataset,false
default,bronze_order_reviews_dataset,false
default,bronze_orders_dataset,false
default,bronze_product_category_name_translation,false
default,bronze_products_dataset,false
default,bronze_sellers_dataset,false
default,customers,false


TOTAL TABLES: 26


In [0]:

tables = spark.sql("SHOW TABLES IN workspace.default")

display(
    tables.select("tableName").orderBy("tableName")
)


tableName
bronze_customers_dataset
bronze_geolocation_dataset
bronze_order_items_dataset
bronze_order_payments_dataset
bronze_order_reviews_dataset
bronze_orders_dataset
bronze_product_category_name_translation
bronze_products_dataset
bronze_sellers_dataset
customers


In [0]:

project_tables = spark.sql("SHOW TABLES IN workspace.default")

project_tables = project_tables.filter(
    F.col("tableName").startswith("bronze_") |
    F.col("tableName").startswith("silver_") |
    F.col("tableName").startswith("gold_")
)

print("PROJECT TABLES:", project_tables.count())

display(project_tables.orderBy("tableName"))


PROJECT TABLES: 25


database,tableName,isTemporary
default,bronze_customers_dataset,false
default,bronze_geolocation_dataset,false
default,bronze_order_items_dataset,false
default,bronze_order_payments_dataset,false
default,bronze_order_reviews_dataset,false
default,bronze_orders_dataset,false
default,bronze_product_category_name_translation,false
default,bronze_products_dataset,false
default,bronze_sellers_dataset,false
default,gold_category_summary,false
